In [58]:
import pandas as pd
import pymongo
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
import pickle

In [59]:
# df = pd.read_csv("training_data.csv", encoding='latin-1')
client = pymongo.MongoClient("mongodb://localhost:27017")
db = client["purePicks"]
col = db["products"]
products = []

for item in col.find():
    products.append(item)

df = pd.DataFrame(products)
df = df.drop(columns=['_id'])
df = df.sort_values(by='id')

In [60]:
le_type = LabelEncoder()
df['type_encoded'] = le_type.fit_transform(df['type'])

# le_origin = LabelEncoder()
# df['type_origin'] = le_origin.fit_transform(df['origin'])

# le_organic = LabelEncoder()
# df['type_organic'] = le_organic.fit_transform(df['organic_or_inorganic'])

scaler = StandardScaler()
df['price_scaled'] = scaler.fit_transform(df[['price']])

# X = df[['type_encoded', 'type_origin', 'type_organic', 'price_scaled']]
X = df[['type_encoded', 'price_scaled']]

neighbors = NearestNeighbors(n_neighbors = 5)
neighbors_model = neighbors.fit(X)

distances, indices = neighbors_model.kneighbors(X[df['id'] == 2])
# print(indices.flatten())

neartest_neighbors = df.iloc[indices[0]]

# print(neartest_neighbors[['id','item','type','price','origin', 'organic_or_inorganic']])
print(neartest_neighbors[['id','item','type','price']])

    id                item   type  price
2    2    Honeycrisp Apple  Fruit   0.69
1    1  Granny Smith Apple  Fruit   0.99
21  22              Tomato  Fruit   0.21
28  28          Fuji Apple  Fruit   0.19
27  29              Lemons  Fruit   0.05


In [61]:
file_path = "trained_model.pkl"

# model_data = {
#     "model": neighbors_model,
#     "le_type": le_type,
#     "le_origin": le_origin,
#     "le_organic": le_organic,
#     "scaler": scaler
# }

model_data = {
    "model": neighbors_model,
    "le_type": le_type,
    "scaler": scaler
}

with open(file_path, 'wb') as file:
    pickle.dump(model_data, file)
